# DSPy Fundamentals: Signatures, Modules, and Predictions

The rest of this lab puts DSPy *optimizers* to work — GEPA, MIPROv2, COPRO, SIMBA — on programs built from signatures and modules. This notebook introduces those building blocks first, so the optimizer notebooks (01 onward) read as *tuning* something you already understand rather than magic.

By the end you should be able to say, in one breath, what a **signature**, a **module**, and a **prediction** are, and name the handful of modules DSPy ships.


In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



## Configure the language model

One language model backs every module below. DSPy talks to it through LiteLLM, so the same code reaches hosted OpenAI or a self-hosted server — only the `dspy.LM(...)` line changes (see the comment).

In [2]:
from rich import print as rprint
# LLM_PROVIDER in .env selects the backend:
#   supportvectors -> SV cluster (gpt-oss-20b)
#   anything else  -> hosted OpenAI (gpt-5-mini; needs OPENAI_API_KEY)
import dspy
from svlearn_dspy.llm import make_lm

lm = make_lm()
dspy.configure(lm=lm)

## The mental model: *program*, don't *prompt*

DSPy's premise is that you should **program** a language model, not hand-write its prompt strings. Three pieces make that work, and they are the whole of this notebook:

- A **Signature** *declares* a task as typed inputs → typed outputs (e.g. `question -> answer`). It says *what* you want, not *how* to ask for it.
- A **Module** is a *strategy* for satisfying a signature with the LM — a single call (`Predict`), a call that reasons first (`ChainOfThought`), a tool-using loop (`ReAct`), and so on. Modules compose into larger programs.
- A **Prediction** is the typed result a module returns: named output fields you read as attributes.

Crucially, a signature's instruction (its docstring) and its few-shot examples are **parameters**, not something you freeze by hand. That is exactly what the optimizers in the next notebooks rewrite.

## 1. Signatures — the inline form

The quickest signature is a string: `"<inputs> -> <outputs>"`. Here one input `question` and one output `response`. `dspy.Predict` wraps it into the simplest module — one LM call that fills the output fields.

In [3]:
predict = dspy.Predict("question -> response")
pred = predict(question="Why is the sky blue? Answer in one sentence.")
rprint(pred.response)

The sky appears blue because molecules and small particles in Earth's atmosphere scatter shorter (blue) wavelengths
of sunlight much more strongly than longer (red) wavelengths, sending blue light in all directions to our eyes.

DSPy wrote the actual prompt for you from the signature. `inspect_history` shows the last call — the system/user messages DSPy built, and how it asked the model to format its answer.

In [4]:
dspy.inspect_history(n=1)





[2026-11-05T20:03:06.945801]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `response`.


User message:

[[ ## question ## ]]
Why is the sky blue? Answer in one sentence.

Respond with the corresponding output fields, starting with the field `[[ ## response ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## response ## ]]
The sky appears blue because molecules and small particles in Earth's atmosphere scatter shorter (blue) wavelengths of sunlight much more strongly than longer (red) wavelengths, sending blue light in all directions to our eyes.
[[ ## completed ## ]]







## 2. Modules — a strategy for the same signature

Swap the *module* and the signature stays the same. `dspy.ChainOfThought` adds a hidden `reasoning` output field, so the model thinks step by step before the answer — same inputs, same declared output, a different strategy for getting there.

In [6]:
cot = dspy.ChainOfThought("question -> response")
c = cot(question="A bat and ball cost $1.10. The bat costs $1 more than the ball. How much is the ball?")
rprint("reasoning:", c.reasoning)
rprint("response :", c.response)

reasoning: Let the ball cost x dollars. Then the bat costs x + 1 dollar. Their sum:
x + (x + 1) = 1.10
2x + 1 = 1.10
2x = 0.10
x = 0.05

So the ball costs $0.05 (five cents).

response : $0.05 (5¢)

## 3. Predictions — the typed result

A module returns a `dspy.Prediction`: each output field is an attribute (`pred.response`, `c.reasoning`). Output fields can carry **types**, and DSPy coerces and validates the model's text into them. Add `: float` to an output and you get a number back, not a string.

In [8]:
math = dspy.ChainOfThought("question -> answer: float")
p = math(question="What is 37 * 43?")
rprint("type:", type(p.answer).__name__, "| answer:", p.answer)
rprint("reasoning:", p.reasoning)

type: float | answer: 1591.0

reasoning: Compute 37 × 43: 37 × 40 = 1480 and 37 × 3 = 111. Sum = 1480 + 111 = 1591.

## 4. Class-based signatures — the form the rest of the lab uses

Inline strings are fine for one line; for real tasks you write a **class**. The class docstring becomes the task *instruction*, and `InputField` / `OutputField` carry types and optional `desc` hints. Typed outputs — `Literal[...]`, lists, dicts, floats — turn the LM's text into validated Python values.

This is exactly the shape notebook 02 uses for its urgency / sentiment / category classifiers.

In [9]:
from typing import Literal

class Classify(dspy.Signature):
    """Classify the subject of the given sentence."""

    sentence: str = dspy.InputField()
    subject: Literal["Health", "Math", "Politics"] = dspy.OutputField()
    confidence: float = dspy.OutputField(desc="0.0-1.0, how sure you are")

classify = dspy.Predict(Classify)
for s in [
    "Regularly monitoring lipid levels is important.",
    "A one-to-one and onto function is invertible.",
    "Opinions keep polarising; neutral media is hard to find.",
]:
    pred = classify(sentence=s)
    rprint(f"{pred.subject:10} ({pred.confidence:.2f})  <-  {s}")

Health     (0.98)  <-  Regularly monitoring lipid levels is important.

Math       (1.00)  <-  A one-to-one and onto function is invertible.

Politics   (0.92)  <-  Opinions keep polarising; neutral media is hard to find.

> The docstring (*"Classify the subject…"*) and each field's `desc` are the **starting prompt**. When an optimizer runs in the next notebooks, this instruction is one of the things it rewrites — together with a set of few-shot examples it chooses for you.

In [10]:
from typing import Literal
import dspy
from rich import print as rprint


# 1. Define the task and allowed outputs.
class MatchTransactionToStore(dspy.Signature):
    """
    Match a credit card transaction description to a store in our catalog.

    Example store catalog:
    - store_nordstrom: Nordstrom. Known name: NORDSTROM.
    - store_nordstrom_rack: Nordstrom Rack.
      Known names: NORDSTROM RACK, NORDSTROMRACK.
    - store_target: Target. Known name: TARGET.

    Rules:
    - Ignore capitalization, branch numbers, and channel labels like ONLINE.
    - Nordstrom and Nordstrom Rack are separate catalog entities.
    - A payment processor prefix alone does not identify the store.
    - Base your decision only on evidence in the transaction description.
    - Return unknown if the merchant is outside this catalog or the
      description provides insufficient evidence.
    """

    transaction_description: str = dspy.InputField(
        desc="The raw merchant description from a credit card transaction."
    )

    store_id: Literal[
        "store_nordstrom",
        "store_nordstrom_rack",
        "store_target",
        "unknown",
    ] = dspy.OutputField()

    evidence: str = dspy.OutputField(
        desc="Briefly explain the evidence supporting your decision in English."
    )


# 2. Use the language model already configured with dspy.configure().
matcher = dspy.Predict(MatchTransactionToStore)


# 3. Synthetic examples with expected labels.
test_cases = [
    ("NORDSTROM #0831", "store_nordstrom"),
    ("NORDSTROM ONLINE", "store_nordstrom"),
    ("NORDSTROMRACK.COM", "store_nordstrom_rack"),
    ("TARGET T-1234", "store_target"),
    ("PAYPAL *XYZSHOP", "unknown"),
]


# 4. Run predictions and check correctness.
correct = 0

for description, expected in test_cases:
    prediction = matcher(transaction_description=description)

    is_correct = prediction.store_id == expected
    correct += int(is_correct)

    rprint({
        "transaction": description,
        "predicted_store": prediction.store_id,
        "expected_store": expected,
        "correct": is_correct,
        "evidence": prediction.evidence,
    })
    print("-" * 60)

print(f"Accuracy on synthetic examples: {correct}/{len(test_cases)}")

{
    'transaction': 'NORDSTROM #0831',
    'predicted_store': 'store_nordstrom',
    'expected_store': 'store_nordstrom',
    'correct': True,
    'evidence': 'Merchant name "NORDSTROM" appears in the description (branch number ignored), matching the 
Nordstrom catalog entry.'
}

------------------------------------------------------------


{
    'transaction': 'NORDSTROM ONLINE',
    'predicted_store': 'store_nordstrom',
    'expected_store': 'store_nordstrom',
    'correct': True,
    'evidence': 'The description contains "NORDSTROM" (the "ONLINE" channel label is ignored), matching the 
Nordstrom store.'
}

------------------------------------------------------------


{
    'transaction': 'NORDSTROMRACK.COM',
    'predicted_store': 'store_nordstrom_rack',
    'expected_store': 'store_nordstrom_rack',
    'correct': True,
    'evidence': 'Description contains "NORDSTROMRACK" (matches the known name "Nordstrom Rack").'
}

------------------------------------------------------------


{
    'transaction': 'TARGET T-1234',
    'predicted_store': 'store_target',
    'expected_store': 'store_target',
    'correct': True,
    'evidence': 'The description includes "TARGET", which matches the known name for Target.'
}

------------------------------------------------------------


{
    'transaction': 'PAYPAL *XYZSHOP',
    'predicted_store': 'unknown',
    'expected_store': 'unknown',
    'correct': True,
    'evidence': 'Description shows a PayPal payment to "XYZSHOP" and does not contain the known retailer names 
(NORDSTROM, NORDSTROM RACK, or TARGET). The processor prefix "PAYPAL" alone is insufficient to identify one of the 
catalog stores.'
}

------------------------------------------------------------
Accuracy on synthetic examples: 5/5


In [11]:
prediction = matcher(
    transaction_description="NORDSTROM RACK #0218"
)

print("Matched store:", prediction.store_id)
print("Evidence:", prediction.evidence)

# Inspect the actual prompt and model response.
dspy.inspect_history(n=1)

Matched store: store_nordstrom_rack
Evidence: The description contains "NORDSTROM RACK", which directly matches the known merchant name for Nordstrom Rack; branch number ignored.




[2026-11-05T20:40:19.712866]

System message:

Your input fields are:
1. `transaction_description` (str): The raw merchant description from a credit card transaction.
Your output fields are:
1. `store_id` (Literal['store_nordstrom', 'store_nordstrom_rack', 'store_target', 'unknown']): 
2. `evidence` (str): Briefly explain the evidence supporting your decision in English.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## transaction_description ## ]]
{transaction_description}

[[ ## store_id ## ]]
{store_id}        # note: the value you produce must exactly match (no extra characters) one of: store_nordstrom; store_nordstrom_rack; store_target; unknown

[[ ## evidence ## ]]
{evidence}

[[ ## completed ## ]]
In adhering to this structure, your objective i

## 5. The module zoo

DSPy ships a small set of modules. All take a signature; they differ in *strategy*:

| Module | Strategy |
|---|---|
| `dspy.Predict` | one LM call that fills the output fields |
| `dspy.ChainOfThought` | adds a `reasoning` field — think before answering |
| `dspy.ReAct` | a reason–act loop that can call **tools** (plain Python functions) until it can answer |
| `dspy.ProgramOfThought` | writes and runs Python code to reach the answer (needs a code sandbox) |
| `dspy.BestOfN` / `dspy.Refine` | run a module several times and keep the best / retry until a reward threshold |
| `dspy.Module` (subclass) | **your own** program composing the above in a `forward()` method |

`ReAct` is the one worth running once. Give it ordinary Python functions as tools; DSPy exposes their names, signatures and docstrings to the model, which decides when to call them.

In [12]:
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

react = dspy.ReAct("question -> answer: float", tools=[multiply])
pred = react(question="What is 5 multiplied by the Hardy-Ramanujan number, 1729?")
rprint("answer:", pred.answer)

answer: 8645.0

In [9]:
dspy.inspect_history(n=1)   # see the tool call ReAct made, and how it finished





[2026-11-05T15:38:47.588771]

System message:

Your input fields are:
1. `question` (str): 
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (float):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
What is 5 multiplied by the Hardy-Ramanujan number, 1729?

[[ ## trajectory ## ]]
[[ ## thought_0 ## ]]
I need to compute 5 multiplied by 1729. I'll let the multiply tool perform the calculation.

[[ ## tool_name_0 ## ]]
multiply

[[ ## tool_args_0 ## ]]
{"a": 5, "b": 1729}

[[ ## observation_0 ## ]]
8645.0

[[ ## thought_1 ## ]]
I have

The two modules not run above:

- **`dspy.ProgramOfThought`** takes the same signature but has the model *write Python* and executes it for the answer — strong for arithmetic and data wrangling, but it needs a code sandbox installed, so it is left as a mention here.
- **`dspy.Refine`** and **`dspy.BestOfN`** wrap another module: they run it up to *N* times, score each result with a `reward_fn`, and return the best (or the first above a threshold). They are a run-time way to trade tokens for quality — separate from the *optimizers* in the next notebooks, which change the program's prompts once, offline.

## 6. Composing modules into a program

Real programs subclass `dspy.Module`, build sub-modules in `__init__`, and wire them in `forward()`, returning a `dspy.Prediction`. Below, a two-stage triage: summarise a message, then classify its urgency from the summary. Notebook 02's `FacilitySupportAnalyzerMM` is the same pattern with three classifiers.

In [10]:
class Triage(dspy.Module):
    def __init__(self):
        self.summarize = dspy.Predict("message -> summary")
        self.rate = dspy.ChainOfThought("summary -> urgency: Literal['low', 'medium', 'high']")

    def forward(self, message: str):
        summary = self.summarize(message=message).summary
        urgency = self.rate(summary=summary).urgency
        return dspy.Prediction(summary=summary, urgency=urgency)

triage = Triage()
out = triage(message="The walk-in freezer in the main kitchen has been off since last night and the food is thawing. Please help!")
rprint("summary:", out.summary)
rprint("urgency:", out.urgency)

summary: Walk‑in freezer in main kitchen has been off since last night; food is thawing. Immediate assistance 
required.

urgency: high

## 7. The optimization loop: examples, a metric, evaluate, compile, re-evaluate

Everything so far was hand-written. The point of DSPy — and of notebooks 01 onward — is to let an **optimizer** improve a program for you. The loop has five pieces, and they are the vocabulary notebook 02 opens with, so we run a tiny version of it here on the `Classify` signature from above.

### Examples and a dataset

A **`dspy.Example`** is one labelled data point: input fields plus the fields you wish the program would produce. `.with_inputs(...)` marks which fields are *inputs* — the rest are the labels the metric checks against. You split them as usual: a **trainset** the optimizer learns from, and a held-out **devset** you score on.

In [11]:
gold = [
    ("Regular aerobic exercise lowers resting heart rate.", "Health"),
    ("The derivative of sin(x) is cos(x).", "Math"),
    ("The senate passed the budget bill after a long debate.", "Politics"),
    ("A balanced diet reduces the risk of type-2 diabetes.", "Health"),
    ("A prime number has exactly two distinct divisors.", "Math"),
    ("Voter turnout was the highest in three decades.", "Politics"),
    ("Vaccination builds immunity before exposure.", "Health"),
    ("The integral of 1/x is the natural logarithm.", "Math"),
    ("Coalition talks collapsed over the tax proposal.", "Politics"),
    ("Sleep deprivation impairs the immune response.", "Health"),
    ("A matrix is singular when its determinant is zero.", "Math"),
    ("The minister resigned amid the funding scandal.", "Politics"),
]
data = [dspy.Example(sentence=s, subject=y).with_inputs("sentence") for s, y in gold]
trainset, devset = data[:6], data[6:]
rprint(f"{len(trainset)} train, {len(devset)} dev")
rprint("one example — inputs:", trainset[0].inputs(), "| label:", trainset[0].subject)

6 train, 6 dev

one example — inputs: Example({'sentence': 'Regular aerobic exercise lowers resting heart rate.'}) 
(input_keys={'sentence'}) | label: Health

### A metric

A **metric** is any callable `metric(example, prediction, trace=None)` that returns a number (or a bool). It says how good one prediction is. The *same* metric drives both evaluation (scoring a program) and optimization (the signal the optimizer climbs). Ours is plain subject accuracy.

> The lab's real metrics go further — notebook 02 returns a score *and* a sentence of textual feedback, which is the extra signal GEPA's reflection LM reads. Same shape, richer return.

In [12]:
def subject_match(example, prediction, trace=None):
    return example.subject == prediction.subject

baseline = dspy.Predict(Classify)   # the Classify signature from section 4

### Evaluate the baseline

`dspy.Evaluate` runs a program over the devset and averages the metric. This is the number every optimizer is trying to move.

In [13]:
evaluator = dspy.Evaluate(devset=devset, metric=subject_match, num_threads=4, display_progress=True)
before = evaluator(baseline).score
rprint(f"baseline dev accuracy: {before:.1f}%")

Average Metric: 6.00 / 6 (100.0%): 100%|██████████| 6/6 [00:02<00:00,  2.27it/s]

2026/11/05 15:38:50 INFO dspy.evaluate.evaluate: Average Metric: 6 / 6 (100.0%)


baseline dev accuracy: 100.0%

### Compile (optimize), then re-evaluate

**Compiling** is DSPy's word for optimizing: you hand an optimizer your program, the trainset and the metric, and it returns a *new* program with better prompts. The simplest optimizer, `BootstrapFewShot`, runs your program on the trainset, keeps the traces where the metric passes, and attaches a few of them as **few-shot examples** — it discovers demonstrations instead of you writing them. Then we score the compiled program the same way and compare.

In [14]:
optimizer = dspy.BootstrapFewShot(metric=subject_match, max_bootstrapped_demos=2, max_labeled_demos=2)
compiled = optimizer.compile(baseline, trainset=trainset)

after = evaluator(compiled).score
rprint(f"baseline : {before:.1f}%")
rprint(f"compiled : {after:.1f}%")

 33%|███▎      | 2/6 [00:02<00:04,  1.12s/it]


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Average Metric: 6.00 / 6 (100.0%): 100%|██████████| 6/6 [00:02<00:00,  2.30it/s]

2026/11/05 15:38:55 INFO dspy.evaluate.evaluate: Average Metric: 6 / 6 (100.0%)


baseline : 100.0%

compiled : 100.0%

Look at what compiling actually changed — the compiled program now carries chosen demonstrations in its prompt:

In [15]:
demos = compiled.predictors()[0].demos
rprint(f"{len(demos)} few-shot demo(s) attached by the optimizer")
if demos:
    rprint("example demo:", {"sentence": demos[0].get("sentence"), "subject": demos[0].get("subject")})
dspy.inspect_history(n=1)   # the compiled prompt now includes those demos

2 few-shot demo(s) attached by the optimizer

example demo:
{'sentence': 'Regular aerobic exercise lowers resting heart rate.', 'subject': 'Health'}





[2026-11-05T15:38:55.176619]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `subject` (Literal['Health', 'Math', 'Politics']): 
2. `confidence` (float): 0.0-1.0, how sure you are
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## subject ## ]]
{subject}        # note: the value you produce must exactly match (no extra characters) one of: Health; Math; Politics

[[ ## confidence ## ]]
{confidence}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the subject of the given sentence.


User message:

[[ ## sentence ## ]]
Regular aerobic exercise lowers resting heart rate.


Assistant message:

[[ ## subject ## ]]
Health

[[ ## confidence ## ]]
0.99

[[ ## completed ## ]]


User message:

[[ ## sentence ## ]]
The derivative of sin(x) is cos(x).


Assistant

On a three-class task a strong model may already score at or near the ceiling, so the before/after gap here can be small — the point is the **mechanism**, not the win. The gains get real on the harder tasks in notebooks 02–10, where there is room to climb and the optimizers do far more than pick demos: GEPA, MIPROv2, COPRO and SIMBA also **rewrite the instruction** (the signature's docstring), guided by the metric — and, for GEPA, by a stronger reflection LM reading the traces. The loop, though — examples, a metric, evaluate, compile, re-evaluate — is exactly the one you just ran.

## Where this goes next

Everything the optimizers tune lives in the pieces above: the **instruction** in a signature's docstring and field descriptions, and the **few-shot examples** DSPy prepends to a module's prompt. An optimizer takes your program (a `dspy.Module`), a dataset of `dspy.Example`s, and a **metric**, and searches for instructions and examples that score higher — without you editing a single prompt string.

Notebook **01** lays out the optimizer landscape; **02–05** then run GEPA, MIPROv2, COPRO and SIMBA on the very signature-and-module style you just met.